# CLYVO Copilot — Avaliação do modelo preditivo (C2)

FIAP 2026 · Disruptive Architectures: IoT, IoB & Generative IA · Sprint 3

Este notebook avalia a camada preditiva do CLYVO Copilot:

- **IsolationForest** (não supervisionado) — anomalia da telemetria vs. baseline individual
- **RandomForest** (supervisionado) — `P(precisa de atenção veterinária em 7 dias)`
- **Regressão Logística** — baseline de comparação

> Os dados são **sintéticos, com seed fixa** (`data/generate_dataset.py`). O objetivo é demonstrar a arquitetura e o processo de avaliação, não acurácia clínica real.

Pré-requisitos: `python data/generate_dataset.py --reset` e `python -m ai_core.risk_model train` (o `dashboard_server.py` faz isso automaticamente na primeira execução).

In [ ]:
import json, os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, ConfusionMatrixDisplay, classification_report

from ai_core.features import FEATURE_ORDER, TELEMETRY_FEATURES, FEATURE_LABELS
from ai_core.risk_model import RiskModel, load_metrics, DATASET_PATH, LABEL

plt.rcParams['figure.dpi'] = 110
os.makedirs('docs/img', exist_ok=True)

metrics = load_metrics()
df = pd.read_csv(DATASET_PATH)
print('Treinado em:', metrics['trained_at'])
print('Dataset:', metrics['dataset'], '|', metrics['n_rows'], 'linhas |', f"{metrics['positive_rate']:.1%} positivos")
df.head()

## 1. Distribuição do dataset

Cada linha é um *episódio*: uma janela recente de telemetria (z-scores e tendências em σ do baseline individual) + contexto clínico, rotulado com `needs_attention_7d`. A coluna `pattern` é apenas auxiliar (não entra no modelo).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
df[LABEL].value_counts().sort_index().plot.bar(ax=axes[0], color=['#3fb950', '#f85149'])
axes[0].set_title('Rótulo: precisa de atenção em 7 dias'); axes[0].set_xticklabels(['não (0)', 'sim (1)'], rotation=0)
df['pattern'].value_counts().plot.barh(ax=axes[1], color='#00b4ff'); axes[1].set_title('Padrão gerador (auxiliar)')
plt.tight_layout(); plt.show()

df.groupby(LABEL)[TELEMETRY_FEATURES].mean().round(2)

## 2. Métricas no conjunto de teste (25 %, estratificado)

Métricas calculadas em `ai_core/risk_model.py::train` e salvas em `ai_core/trained/metrics.json`.

In [ ]:
tab = pd.DataFrame({
    'RandomForest': metrics['random_forest'],
    'Regressão Logística (baseline)': metrics['logistic_regression'],
}).drop(index='confusion_matrix')
display(tab)
print('IsolationForest — AUC como detector (só telemetria, treinado só com normais):', metrics['isolation_forest']['roc_auc_as_detector'])
print(metrics['score_formula'])

## 3. Matriz de confusão e curvas ROC

Reproduzimos a mesma divisão treino/teste (`random_state=42`) usada no treino para avaliar o modelo salvo.

In [ ]:
model = RiskModel.load()
X = df[FEATURE_ORDER].to_numpy(float); y = df[LABEL].to_numpy(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

rf_prob = model.clf.predict_proba(X_te)[:, 1]
raw = -model.iso.score_samples(X_te[:, model.tel_idx])
anom = np.clip((raw - model.q01) / max(model.q99 - model.q01, 1e-9), 0, 1)
risk = 100 * (0.65 * rf_prob + 0.35 * anom)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_te, (rf_prob >= 0.5).astype(int)), display_labels=['não', 'sim']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('RandomForest — matriz de confusão (limiar 0,5)')

for name, score in [('RandomForest', rf_prob), ('IsolationForest (anomalia)', anom), ('risk_score combinado', risk / 100)]:
    fpr, tpr, _ = roc_curve(y_te, score)
    axes[1].plot(fpr, tpr, label=f'{name} — AUC {roc_auc_score(y_te, score):.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=0.8); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('Curvas ROC'); axes[1].legend(fontsize=8)

axes[2].hist(risk[y_te == 0], bins=25, alpha=0.7, color='#3fb950', label='não precisa')
axes[2].hist(risk[y_te == 1], bins=25, alpha=0.7, color='#f85149', label='precisa de atenção')
for cut, lab in [(30, 'Moderado'), (60, 'Alto'), (80, 'Crítico')]:
    axes[2].axvline(cut, color='gray', ls=':', lw=0.8); axes[2].text(cut + 1, axes[2].get_ylim()[1] * 0.9, lab, fontsize=7, color='gray')
axes[2].set_title('Distribuição do risk_score por classe'); axes[2].set_xlabel('risk_score'); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig('docs/img/avaliacao_modelo.png', bbox_inches='tight'); plt.show()

print(classification_report(y_te, (rf_prob >= 0.5).astype(int), target_names=['não precisa', 'precisa de atenção']))

## 4. Explicabilidade — importância das features

As features de atividade (`act_z`) e temperatura (`temp_z`, `temp_slope`) dominam, coerente com o modelo gerador e com a literatura clínica: queda de atividade costuma anteceder alterações de temperatura.

In [ ]:
imp = pd.Series(metrics['feature_importance']).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#00b4ff' if f in TELEMETRY_FEATURES else '#a371f7' for f in imp.index]
ax.barh([FEATURE_LABELS[f] for f in imp.index], imp.values, color=colors)
ax.set_title('RandomForest — importância das features (azul: telemetria · roxo: clínico)')
plt.tight_layout(); plt.savefig('docs/img/feature_importance.png', bbox_inches='tight'); plt.show()

## 5. Do modelo à decisão — pets do banco

O `pipeline.py` junta C1 (features do banco), C2 (este modelo), C3 (regras), C4 (serviços), IoB e C5 (redação). Abaixo, o resultado em modo template (offline) para os pets-vitrine do seed.

In [ ]:
from ai_core import db, pipeline
rows = []
for p in db.list_pets():
    ins = pipeline.analyze_pet(p['id'], llm_mode='offline', persist=False)
    r = ins['risk']
    rows.append({'pet': ins['pet_name'], 'espécie': ins['species'], 'risk_score': r['risk_score'], 'nível': r['risk_level'],
                 'P(7d)': r['prob_attention_7d'], 'anomalia': r['anomaly_score'],
                 'fator principal': (r['top_factors'][0]['label'] + ' ' + r['top_factors'][0]['value_text']) if r['top_factors'] else '—',
                 'próxima ação': (ins['actions'][0]['priority_label'] + ' · ' + ins['actions'][0]['title']) if ins['actions'] else 'em dia',
                 'serviço': ins['services'][0]['name'] if ins['services'] else '—',
                 'tutor (IoB)': ins['tutor_profile']['engagement']})
pd.DataFrame(rows).sort_values('risk_score', ascending=False).reset_index(drop=True)

## 6. Conclusões

- O **RandomForest** supera a regressão logística (baseline) em AUC e F1, indicando interações não lineares entre sinais (ex.: temperatura ↑ **e** atividade ↓).
- O **IsolationForest**, mesmo sem rótulos, separa razoavelmente os episódios — viável como detector desde o primeiro dia (cold start).
- O `risk_score` combinado concentra os saudáveis abaixo de 30 e os casos com sinais fortes acima de 60, o que sustenta os cortes Baixo/Moderado/Alto/Crítico usados na fila de triagem.
- **Limitação**: dados sintéticos. Em produção, o rótulo viria do desfecho real (consulta em 7 dias / diagnóstico registrado) e o modelo seria reavaliado com validação temporal.